# G1 — Kaggle background commit

Upload `g1_code.zip` built by `make_bundle.sh`; private code stays in a private dataset/Drive folder.
CPU PPO only. Repository pins: MuJoCo 3.3.7, SB3 2.3.2, Gymnasium 0.29.1,
PyTorch 2.2.2 CPU, NumPy 1.26.4; isolated Python 3.12, no kernel restart.
Runs the 600-second smoke, then the selected G1 seeds with `--jobs=os.cpu_count()`.
Default split: Kaggle **0,1,2**, Colab **3,4**; merge all five before final evaluation.
Training checkpoints remain every **5 minutes** (frozen config); a verified recovery ZIP and
learning curves are refreshed every **10 minutes**, and at exit. Wall-clock intervals include
completion of an optimizer update and file I/O.

**Seed wall time is unlimited by default.** Set `SEED_WALL_CAP_MIN` for an optional cumulative
per-seed cap. Missing readiness evidence or a failed/interrupted pilot still stops the queue.
This notebook does not create readiness flags or change budgets/rewards. A stopped run exports
its reason and partial artifacts; it is not a completed result. See `CLOUD_README.md` for the
4-vCPU **ESTIMATE**, persistence/resume steps, and prerequisites.


## 1. Parameters, paths and input

Edit only this cell if automatic input discovery is ambiguous. Keep the same source bundle,
seed list, CPU count, and run directory on resume. Do not run two notebooks against one run directory.
`READINESS_JSON` is optional evidence supplied separately; leave blank when unresolved.


In [ ]:
SEEDS = [0, 1, 2]  # final seeds for this part; keep unchanged on resume
from pathlib import Path
import time
SESSION_STARTED = time.monotonic()  # include unpacking/install time in the global budget
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working')
SCRATCH = Path('/tmp/g1-cloud')  # venv/source never enter saved outputs
CODE_SOURCE = ''  # optional zip path, or expanded bundle manifest/directory
RESUME_SOURCE = ''  # optional g1-recovery.zip path or expanded g1-run directory
READINESS_JSON = ''  # optional real host-readiness.json path; never fabricated
MAX_HOURS = 11.5  # global soft stop, including setup; checkpoint + resume note + export
SEED_WALL_CAP_MIN = None  # optional finite positive minutes; default unlimited
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SCRATCH.mkdir(parents=True, exist_ok=True)


## 2. Print CPUs, unpack source, and restore retained outputs


In [ ]:
import hashlib
import json
import os
from pathlib import Path
import shutil
import stat
import zipfile

JOBS = os.cpu_count() or 1
print(f"Logical CPUs: {JOBS}; G1 --jobs {JOBS}", flush=True)
for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
             'VECLIB_MAXIMUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = ''
os.environ['MPLBACKEND'] = 'Agg'


def safe_unpack(archive, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            target = (destination/member.filename).resolve()
            if not target.is_relative_to(destination) or stat.S_ISLNK(member.external_attr >> 16):
                raise ValueError(f'Unsafe archive path: {member.filename}')
        z.extractall(destination)


def choose(paths, label):
    paths = sorted(set(paths))
    if len(paths) != 1:
        raise RuntimeError(f'Set {label} explicitly; found {len(paths)} candidates: {paths}')
    return paths[0]


if CODE_SOURCE:
    source = Path(CODE_SOURCE)
else:
    archives = list(INPUT_ROOT.rglob('g1_code.zip'))
    source = choose(archives or list(INPUT_ROOT.rglob('G1_BUNDLE_MANIFEST.json')), 'CODE_SOURCE')
if source.is_dir():
    source = source/'G1_BUNDLE_MANIFEST.json'
source_hash = hashlib.sha256(source.read_bytes()).hexdigest()
REPO = SCRATCH/('g1-code-'+source_hash[:16])
REPO.mkdir(parents=True, exist_ok=True)
if source.suffix == '.zip':
    safe_unpack(source, REPO)
else:  # Kaggle may expand uploaded dataset zip files automatically.
    manifest = json.loads(source.read_text())
    for relative in [*manifest['files'], 'G1_BUNDLE_MANIFEST.json']:
        target = (REPO/relative).resolve()
        origin = (source.parent/relative).resolve()
        if not target.is_relative_to(REPO.resolve()) or not origin.is_relative_to(source.parent.resolve()):
            raise ValueError(f'Unsafe bundle member: {relative}')
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(origin, target)
manifest = json.loads((REPO/'G1_BUNDLE_MANIFEST.json').read_text())
for relative, digest in manifest['files'].items():
    target = (REPO/relative).resolve()
    if not target.is_relative_to(REPO.resolve()):
        raise ValueError(f'Unsafe manifest path: {relative}')
    if hashlib.sha256(target.read_bytes()).hexdigest() != digest:
        raise ValueError(f'Bundle hash mismatch: {relative}')
print(f"Unpacked {len(manifest['files'])} files; working-tree source HEAD: {manifest['git_head']}")

RUN = OUTPUT_ROOT/'g1-run'
# Restore only into an absent run: never merge two different experiments.
if not RUN.exists():
    if RESUME_SOURCE:
        recovery = Path(RESUME_SOURCE)
    else:
        archives = list(INPUT_ROOT.rglob('g1-recovery.zip'))
        expanded = [p.parent for p in INPUT_ROOT.rglob('cloud-status.json') if p.parent.name == 'g1-run']
        candidates = archives or expanded
        recovery = choose(candidates, 'RESUME_SOURCE') if candidates else None
    if recovery is not None:
        if recovery.is_dir():
            shutil.copytree(recovery, RUN)
        else:
            # Recovery archive is rooted at g1-run/.
            with zipfile.ZipFile(recovery) as z:
                if any(Path(n).parts[0] != 'g1-run' for n in z.namelist()):
                    raise ValueError('Expected a g1-recovery.zip rooted at g1-run/')
            safe_unpack(recovery, OUTPUT_ROOT)
        print(f'Restored prior output from {recovery}')
RUN.mkdir(parents=True, exist_ok=True)
if READINESS_JSON and not (RUN/'host-readiness.json').exists():
    shutil.copyfile(READINESS_JSON, RUN/'host-readiness.json')
# No readiness flags are generated by this notebook.
print('Run/checkpoint directory:', RUN)


## 3. Install pinned CPU dependencies in an isolated Python


In [ ]:
import subprocess
import sys

# Isolate old scientific pins from Kaggle/Colab's preinstalled packages.
# Use Python 3.12 (the repo's version), including on future Python 3.13+ images.
VENV = SCRATCH/'g1-python312'
PYTHON = VENV/'bin/python'
if not PYTHON.exists():
    if sys.version_info[:2] == (3, 12):
        subprocess.run([sys.executable, '-m', 'venv', str(VENV)], check=True)
    else:
        bootstrap = SCRATCH/'g1-uv-bootstrap'
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                        '--target', str(bootstrap), 'uv==0.9.7'], check=True)
        env = dict(os.environ, PYTHONPATH=str(bootstrap))
        subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12.12',
                        '--python-preference', 'only-managed', '--seed', str(VENV)], env=env, check=True)

PINS = ['mujoco==3.3.7', 'stable-baselines3==2.3.2', 'gymnasium==0.29.1',
        'numpy==1.26.4', 'scipy==1.15.3', 'matplotlib==3.10.3']
repo_requirements = (REPO/'sim/requirements.lock').read_text() + (REPO/'sim/rl/requirements.txt').read_text()
assert all(pin in repo_requirements.splitlines() for pin in [*PINS, 'torch==2.2.2'])
subprocess.run([str(PYTHON), '-m', 'pip', 'install', '--disable-pip-version-check',
                'torch==2.2.2+cpu', '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)
subprocess.run([str(PYTHON), '-m', 'pip', 'install', '--disable-pip-version-check',
                *PINS, 'torch==2.2.2+cpu'], check=True)
# Imports and version checks happen in the isolated interpreter; no kernel restart.
probe = """
import importlib.metadata as m, json, torch
expected = {'mujoco': '3.3.7', 'stable-baselines3': '2.3.2', 'gymnasium': '0.29.1',
            'numpy': '1.26.4', 'torch': '2.2.2+cpu', 'scipy': '1.15.3', 'matplotlib': '3.10.3'}
actual = {name: m.version(name) for name in expected}
assert actual == expected, actual
assert torch.version.cuda is None
print(json.dumps(actual, indent=2))
"""
subprocess.run([str(PYTHON), '-c', probe], check=True)
if not (RUN/'source-bundle.json').exists():
    shutil.copyfile(REPO/'G1_BUNDLE_MANIFEST.json', RUN/'source-bundle.json')
freeze = subprocess.check_output([str(PYTHON), '-m', 'pip', 'freeze'], text=True)
import time
(RUN/f'cloud-packages-{time.time_ns()}.txt').write_text(freeze)
if not (RUN/'cloud-packages.txt').exists():
    (RUN/'cloud-packages.txt').write_text(freeze)


## 4. Run smoke, then the selected G1 seeds

Logs stream to `g1-run/cloud-smoke.log` and `cloud-run.log`; the cell prints a heartbeat
each minute. Split parts write `run-manifest.json` and `part-complete.json` after all selected seeds pass.
Merge seeds 0–4 before final evaluation; a trained part is not a final result.
Resume restores optimizer, normalizer, sampler and RNG state from verified checkpoints;
active episodes reset. Interrupted smoke/pilot and failure evidence remain explicit.


In [ ]:
DRIVER_SOURCE = r'''
"""Cloud supervisor; frozen recipe and metrics, configurable runtime limits."""
import argparse
import json
import os
from pathlib import Path
import signal
import subprocess
import sys
import time
import zipfile
from sim.rl.g1.protocol import identity, seed_wall_cap, write_json
from sim.rl.g1.runner import curves
from sim.rl.g1.run_manifest import FINAL_SEEDS, seed_list
from sim.rl.g1.training import newest


def snapshot(run):
    # Only checkpoint generations whose complete.json file hashes verify.
    selected = set()
    warnings = []
    for folder in run.rglob('checkpoints'):
        try:
            selected.add(newest(folder.parent).resolve())
        except RuntimeError as exc:
            warnings.append(str(exc))
    for episodes in run.rglob('episodes.jsonl'):
        try:
            curves(episodes.parent)
        except (ValueError, OSError) as exc:
            warnings.append(f'Curve update deferred: {exc}')
    output = run.parent/'g1-recovery.zip'
    temporary = output.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(run.rglob('*')):
            if not path.is_file() or path.suffix == '.tmp':
                continue
            relative = path.relative_to(run)
            if 'checkpoints' in relative.parts:
                if path.parent.resolve() not in selected:
                    continue
            archive.write(path, path.relative_to(run.parent))
    temporary.replace(output)
    print(f'SNAPSHOT {output}: {len(selected)} verified checkpoints; {warnings}', flush=True)
    return dict(verified_checkpoints=len(selected), warnings=warnings)


def stop_process(process, run):
    # Ask active trainers to finish their current update/checkpoint first.
    (run/'STOP').touch()  # queue barrier also blocks seeds not yet started
    for declaration in run.rglob('declaration.json'):
        (declaration.parent/'STOP').touch()
    until = time.monotonic()+90
    while process.poll() is None and time.monotonic() < until:
        time.sleep(1)
    # Runner may still be evaluating; terminate only this runner process group.
    try:
        os.killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError:
        pass
    try:
        process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        os.killpg(process.pid, signal.SIGKILL)
        process.wait()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--out', type=Path, required=True)
    parser.add_argument('--jobs', type=int, required=True)
    parser.add_argument('--hours', type=float, default=11.5)
    parser.add_argument('--session-elapsed-s', type=float, default=0)
    parser.add_argument('--seeds', type=seed_list, default=FINAL_SEEDS)
    parser.add_argument('--seed-wall-cap-min', type=seed_wall_cap)
    args = parser.parse_args()
    run = args.out.resolve()
    run.mkdir(parents=True, exist_ok=True)
    is_part = args.seeds != FINAL_SEEDS
    marker = run/('part-complete.json' if is_part else 'complete.json')
    done_status = 'TRAINED' if is_part else 'COMPLETE'
    started = time.monotonic()
    deadline = started+args.hours*3600-args.session_elapsed_s
    record = dict(status='RUNNING', jobs=args.jobs, seeds=args.seeds, identity=identity(),
                  checkpoint_seconds=300, recovery_snapshot_seconds=600,
                  seed_wall_cap_min=args.seed_wall_cap_min, soft_stop_hours=args.hours,
                  setup_elapsed_s=args.session_elapsed_s)
    old_path = run/'cloud-status.json'
    if old_path.exists():
        old = json.loads(old_path.read_text())
        if old['identity'] != record['identity'] or old['jobs'] != args.jobs or old.get('seeds', FINAL_SEEDS) != args.seeds:
            raise RuntimeError('Resume source/jobs/seeds differ; retained experiment was not changed')
    # STOP is a transport stop request, not failure evidence. Keep a receipt and
    # clear the request so resuming a cloud time-budget interruption can advance.
    # Never clear failure/watchdog/pilot evidence.
    stopped = list(run.rglob('STOP'))
    if stopped:
        write_json(run/f'cloud-resume-{time.time_ns()}.json',
                   dict(cleared_stop_requests=[str(p.relative_to(run)) for p in stopped]))
        for path in stopped:
            path.unlink()
    process = None
    try:
        if marker.exists():
            if json.loads(marker.read_text())['identity'] != record['identity'] or json.loads(marker.read_text()).get('seeds', FINAL_SEEDS) != args.seeds:
                raise RuntimeError('Completed queue source differs; retained result was not changed')
            record['status'] = done_status
            return
        # Run the real 600s smoke explicitly, once. The full runner reuses it.
        phases = [] if (run/'smoke/smoke.json').exists() else ['smoke']
        if phases and (run/'smoke').exists():
            raise RuntimeError('Interrupted smoke retained; inspect it before starting a fresh run folder')
        phases.append('run')
        for mode in phases:
            if time.monotonic() >= deadline:
                raise TimeoutError('Cloud time budget exhausted')
            destination = run/'smoke' if mode == 'smoke' else run
            command = [sys.executable, '-u', '-m', 'sim.rl.g1.runner', mode,
                       '--out', str(destination), '--jobs', str(args.jobs)]
            if mode == 'run':
                command += ['--seeds', ','.join(map(str, args.seeds))]
            if mode == 'run' and args.seed_wall_cap_min is not None:
                command += ['--seed-wall-cap-min', str(args.seed_wall_cap_min)]
            print('RUN:', ' '.join(command), flush=True)
            record.update(phase=mode, command=command)
            write_json(old_path, record)
            with (run/f'cloud-{mode}.log').open('a', buffering=1) as log:
                process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT,
                                           start_new_session=True)
                next_snapshot = time.monotonic()+600
                next_heartbeat = time.monotonic()+60
                while process.poll() is None:
                    now = time.monotonic()
                    if now >= deadline:
                        raise TimeoutError('Cloud time budget exhausted; retaining PARTIAL checkpoints')
                    if now >= next_snapshot:
                        record['snapshot'] = snapshot(run)
                        write_json(old_path, record)
                        next_snapshot = time.monotonic()+600
                    if now >= next_heartbeat:
                        print(f'{mode}: {(now-started)/60:.1f} min; log: {log.name}', flush=True)
                        next_heartbeat = now+60
                    time.sleep(1)
            record['returncode'] = process.returncode
            if process.returncode:
                tail = (run/f'cloud-{mode}.log').read_text()[-6000:]
                print(tail, flush=True)
                raise RuntimeError(f'G1 {mode} stopped (exit {process.returncode}); inspect retained log')
            process = None
        if not marker.exists():
            raise RuntimeError('Runner exited without a completion receipt')
        record['status'] = done_status
    except BaseException as exc:
        record.update(status='PARTIAL' if isinstance(exc, TimeoutError) else 'STOPPED', error=repr(exc))
        if isinstance(exc, TimeoutError):
            record['resume_note'] = 'RESUME.md'
            (run/'RESUME.md').write_text(
                '# G1 session soft stop\n\n'
                'The global cloud time budget expired. Final results remain incomplete; trained parts still need merging.\n'
                'STOP requests finish the active optimizer update and save verified checkpoints.\n'
                'Reattach g1-recovery.zip or these outputs with the same code bundle, seed list, job count,\n'
                'and run directory; rerun the notebook to resume verified final-seed checkpoints.\n'
                'Counters and cumulative seed wall time are retained. An optional seed cap still applies.\n'
                'Failure/watchdog or interrupted smoke/pilot evidence requires inspection; no automatic retry.\n')
        raise
    finally:
        if process is not None:
            stop_process(process, run)
        record['cloud_elapsed_s'] = time.monotonic()-started
        record['session_elapsed_s'] = args.session_elapsed_s+record['cloud_elapsed_s']
        write_json(old_path, record)
        record['snapshot'] = snapshot(run)
        if record.get('resume_note'):
            write_json(run/'resume-note.json', dict(
                reason=record['error'], phase=record.get('phase'),
                seed_wall_cap_min=args.seed_wall_cap_min, seeds=args.seeds,
                verified_checkpoints=record['snapshot']['verified_checkpoints'],
                session_elapsed_s=record['session_elapsed_s']))
            snapshot(run)  # include the final checkpoint count/resume note in the export
        write_json(old_path, record)
        print(json.dumps(record, indent=2), flush=True)


if __name__ == '__main__':
    main()
'''


In [ ]:
# All training runs in subprocesses from real .py entrypoints, safe with spawn.
DRIVER = SCRATCH/'g1_cloud_driver.py'
DRIVER.write_text(DRIVER_SOURCE)
child_env = dict(os.environ, PYTHONPATH=str(REPO))
command = [str(PYTHON), '-u', str(DRIVER), '--out', str(RUN),
           '--jobs', str(JOBS), '--hours', str(MAX_HOURS),
           '--session-elapsed-s', str(time.monotonic()-SESSION_STARTED),
           '--seeds', ','.join(map(str, SEEDS))]
if SEED_WALL_CAP_MIN is not None:
    command += ['--seed-wall-cap-min', str(SEED_WALL_CAP_MIN)]
RESULT = subprocess.run(command, cwd=REPO, env=child_env)
print('Supervisor exit code:', RESULT.returncode)
print('Status:', (RUN/'cloud-status.json').read_text() if (RUN/'cloud-status.json').exists() else 'See cell output')
# Keep this cell successful so Save & Run All can publish stopped-run outputs too.
# A subset finishes with part-complete.json + run-manifest.json. Merge, then evaluate.


## 5. Inspect status, curves, and saved artifacts


In [ ]:
from IPython.display import display, FileLink, Image
for path in sorted(RUN.rglob('learning-curves.png')):
    print(path.relative_to(OUTPUT_ROOT))
    display(Image(filename=str(path)))
print('Selected seeds:', SEEDS)
print('Part manifest:', RUN/'run-manifest.json')
print('Training receipt:', RUN/'part-complete.json')
print('Merge all five seeds before final evaluation; see CLOUD_README.md.')
print('Recovery / result archive:', OUTPUT_ROOT/'g1-recovery.zip')
if (OUTPUT_ROOT/'g1-recovery.zip').exists():
    display(FileLink(str(OUTPUT_ROOT/'g1-recovery.zip')))
print('Raw output includes logs, episode returns/lengths, reward sums, per-seed receipts, and verified checkpoints.')


Use **Save Version → Save & Run All (Commit)** with output saving enabled; the background
version runs without your browser. Download `g1-recovery.zip` and the `g1-run/` outputs from
the finished version's **Output** tab. `/kaggle/working` is not permanent across fresh sessions:
attach the prior version's outputs or upload the recovery ZIP as a private dataset to resume.
Only artifacts actually saved by Kaggle can survive an abrupt runtime loss. Session limits
vary; roughly 12 hours is an upper bound, not a completion guarantee.

Default split: Kaggle seeds 0,1,2; Colab seeds 3,4. Keep both recovery archives
in separate folders. Follow CLOUD_README.md to merge into one seeds-root and evaluate
on Mac or in a separate Kaggle session using the same code bundle.
